# 🚀 CineGrade AI - Free GPU Backend

Welcome! This notebook provides the 100% free NVIDIA T4 GPU backend required for CineGrade AI.

### Instructions (Quick Start Guide):
1. In the menu at the top, click **Runtime** ➔ **Run all** (or press Ctrl+F9).
2. A popup might warn you that this notebook was not authored by Google. Click **Run anyway**.
3. Scroll down to the bottom cell. It will print out a **Secure Localtunnel URL** (it looks like `https://xyz.loca.lt`).
4. Copy that URL and paste it into the CineGrade web app.

In [ ]:
# @title 1. Setup Environment (Please wait ~60 seconds for this to complete)
import os
from IPython.display import clear_output

print('⏳ Cloning repository and installing dependencies...')
if not os.path.exists('VideoColorGrading'):
    !git clone https://github.com/shameel0505/VideoColorGrading.git
%cd VideoColorGrading

!pip install diffusers==0.21.4 transformers==4.32.0 accelerate==0.22.0 omegaconf==2.3.0 einops==0.6.1 pillow_lut==1.1.0 decord==0.6.0 fastapi uvicorn python-multipart pydantic imageio_ffmpeg rawpy

!npm install -g localtunnel > /dev/null 2>&1

clear_output()
print('✅ Environment setup complete!')


In [ ]:
!pip install pillow-lut
# @title 2. Start GPU Server (Localtunnel)
import subprocess
import time
import os
import urllib.request
print('⏳ Cleaning up old background processes...')
subprocess.run('pkill -f uvicorn', shell=True)
subprocess.run('pkill -f lt', shell=True)
time.sleep(1)
print('⏳ Booting up FastAPI Server on port 8444...')
log_file = open('uvicorn.log', 'w')
fastapi_proc = subprocess.Popen(
    ['uvicorn', 'api:app', '--host', '127.0.0.1', '--port', '8444'],
    stdout=log_file,
    stderr=subprocess.STDOUT
)
server_up = False
for i in range(45):
    time.sleep(1)
    if fastapi_proc.poll() is not None:
        print('\n❌ ERROR: FastAPI crashed! Here is the exact reason why:\n')
        with open('uvicorn.log', 'r') as f:
            print(f.read())
        break
    try:
        urllib.request.urlopen('http://127.0.0.1:8444/api/library')
        server_up = True
        break
    except Exception:
        pass
if not server_up:
    print('❌ Failed to start local server. Halting tunnel creation.')
else:
    print('✅ Server is up! Establishing secure Localtunnel...')
    tunnel_proc = subprocess.Popen(
        ['lt', '--port', '8444'],
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT,
        text=True
    )
    url_found = False
    for line in tunnel_proc.stdout:
        if 'your url is:' in line:
            full_url = line.strip().split('url is: ')[-1]
            print('\n' + '⭐'*30)
            print('\nCOPY THIS EXACT URL AND PASTE IT IN THE WEB APP:')
            print(f'\n--->  {full_url}  <---\n')
            print('⭐'*30 + '\n')
            url_found = True
            break
    if not url_found:
        print('❌ Failed to generate URL. Please restart the cell.')
    try:
        tunnel_proc.wait()
    except KeyboardInterrupt:
        fastapi_proc.terminate()
        print('\nServer stopped.')
